# Session 8 — Extensions and Special Topics

**Course: Event Studies in Finance & Economics**

*Mathis Mourey*

---

Sessions 1 through 7 developed the standard event study toolkit: normal return models, aggregation, parametric and non-parametric tests, cross-sectional regressions, and long-horizon methods. This session extends the methodology in four directions that arise frequently in applied research.

First, we address multi-country event studies, where the researcher must handle multiple exchanges, time zones, currencies, and market models simultaneously. Second, we develop the intraday event study using high-frequency data, which is essential for measuring the impact of events whose timing is known to the minute (monetary policy announcements, earnings releases with precise timestamps). Third, we formalize the treatment of confounding events, which contaminate the event window and bias the measured abnormal return. Fourth, we introduce the conditional event study framework of Prabhala (1997) and Acharya (1988), which accounts for the fact that events are not randomly assigned but are the outcome of an endogenous decision.

Each extension is motivated by a concrete application, developed with the relevant econometric theory, and implemented in code.

**References for this session:**

- Acharya, S. (1988). A Generalized Econometric Model and Tests of a Signalling Hypothesis with Two Discrete Signals. *Journal of Finance*, 43(2), 413--429.
- Barclay, M.J. and Litzenberger, R.H. (1988). Announcement Effects of New Equity Issues and the Use of Intraday Price Data. *Journal of Financial Economics*, 21(1), 71--99.
- Bernanke, B.S. and Kuttner, K.N. (2005). What Explains the Stock Market's Reaction to Federal Reserve Policy? *Journal of Finance*, 60(3), 1221--1257.
- Campbell, C.J. and Wesley, C.E. (1993). Measuring Security Price Performance Using Daily NASDAQ Returns. *Journal of Financial Economics*, 33(1), 73--92.
- McWilliams, A. and Siegel, D. (1997). Event Studies in Management Research: Theoretical and Empirical Issues. *Academy of Management Journal*, 40(3), 626--657.
- Park, N.K. (2004). A Guide to Using Event Study Methods in Multi-Country Settings. *Strategic Management Journal*, 25(7), 655--668.
- Prabhala, N.R. (1997). Conditional Methods in Event Studies and an Equilibrium Justification for Standard Event-Study Procedures. *Review of Financial Studies*, 10(1), 1--38.

## 1. Multi-Country Event Studies

### 1.1 The Problem

When an event affects firms listed on different exchanges in different countries, the standard single-market event study encounters several complications.

**Non-synchronous trading.** If a U.S. policy announcement occurs at 14:00 EST, European markets have already closed. The European price reaction will appear on the following trading day. If the researcher uses the announcement date as the event date for all firms, the European CARs will be measured one day too late.

**Time zone alignment.** Even within the same calendar day, trading hours differ. The Tokyo Stock Exchange closes before London opens, which closes before New York. An event that occurs during U.S. trading hours is a "next-day" event for Asian markets.

**Currency effects.** When aggregating abnormal returns across countries, the researcher must decide whether to work in local currency (measuring the return to a local investor) or in a common currency (measuring the return to an international investor). The difference is the exchange rate return, which can be large and correlated with the event.

**Different market models.** Each country has its own market index, and the appropriate normal return model differs across markets. A firm listed on the TSE should be benchmarked against the TOPIX or Nikkei, not the S&P 500.

### 1.2 Solutions

Park (2004) proposed a practical framework for multi-country event studies. The key elements are:

Adjust the event date for each country based on the time zone of the announcement relative to the local market's trading hours. If the event occurs after market close in country $j$, use $t + 1$ as the event date for firms in country $j$.

Estimate a separate market model for each country, using the local market index as the benchmark. This ensures that the normal return model captures country-specific market dynamics.

Compute abnormal returns in local currency. If the research question concerns the wealth effect for local investors, local-currency returns are appropriate. If the question concerns the wealth effect for a common investor, convert returns using end-of-day exchange rates.

Aggregate using the standard cross-sectional methods of Sessions 3 through 5, treating the multi-country sample as a single cross-section of CARs. The cross-sectional t-test (Session 4) is particularly well-suited because it does not require the abnormal returns to have the same variance across firms (countries).

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# -- Multi-Country Event Study: US-China Trade War Escalation --
# Event: August 23, 2019 — China announces retaliatory tariffs;
#        Trump orders US companies to find alternatives to China.
# This event affected firms globally but at different times due to time zones.

event_date_us = pd.Timestamp('2019-08-23')

firms = pd.DataFrame({
    'ticker': ['AAPL', 'CAT', 'BA', '7203.T', '005930.KS', 'VOW3.DE', 'BABA'],
    'name': ['Apple', 'Caterpillar', 'Boeing', 'Toyota', 'Samsung', 'Volkswagen', 'Alibaba'],
    'country': ['US', 'US', 'US', 'Japan', 'S. Korea', 'Germany', 'US-listed China'],
    'local_market': ['^GSPC', '^GSPC', '^GSPC', '^N225', '^KS11', '^GDAXI', '^GSPC'],
    'event_date': pd.to_datetime([
        '2019-08-23', '2019-08-23', '2019-08-23',  # US: same day
        '2019-08-26', '2019-08-26', '2019-08-26',  # Asia/Europe: next trading day
        '2019-08-23',  # Alibaba US-listed
    ]),
})

print("Multi-Country Sample: Trade War Escalation (Aug 23, 2019)")
print("=" * 75)
print(firms[['ticker', 'name', 'country', 'event_date']].to_string(index=False))
print(f"\nNote: Asian/European event date shifted to next trading day (Aug 26)")

In [ ]:
# -- Download and estimate market models per country --
start_date = '2018-06-01'
end_date = '2020-06-01'
est_length = 250
buffer = 10
event_win = (-5, 5)
event_days = list(range(event_win[0], event_win[1] + 1))

# Download all unique market indices
unique_markets = firms['local_market'].unique()
market_data = {}
for mkt in unique_markets:
    d = yf.download(mkt, start=start_date, end=end_date, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    market_data[mkt] = d['Close'].pct_change().dropna()

# Download all firm data
all_tickers = firms['ticker'].tolist()
d_firms = yf.download(all_tickers, start=start_date, end=end_date, progress=False)
if isinstance(d_firms.columns, pd.MultiIndex):
    firm_prices_all = d_firms['Close']
else:
    firm_prices_all = d_firms[['Close']].rename(columns={'Close': all_tickers[0]})
firm_rets_all = firm_prices_all.pct_change().dropna()

# Estimate market model and compute ARs for each firm using its local market
ar_results = {}
for _, row in firms.iterrows():
    tick = row['ticker']
    mkt_idx = row['local_market']
    edate = row['event_date']

    if tick not in firm_rets_all.columns:
        print(f"  {tick}: data not available, skipping")
        continue

    firm_ret = firm_rets_all[tick]
    mkt_ret = market_data[mkt_idx]
    common = firm_ret.index.intersection(mkt_ret.index)
    firm_ret = firm_ret.loc[common]
    mkt_ret = mkt_ret.loc[common]
    tdays = firm_ret.index

    eidx = tdays.get_indexer([edate], method='ffill')[0]
    ev_s = eidx + event_win[0]
    ev_e = eidx + event_win[1]
    est_e = ev_s - buffer - 1
    est_s = est_e - est_length + 1

    if est_s < 0 or ev_e >= len(tdays):
        print(f"  {tick}: insufficient data")
        continue

    y = firm_ret.iloc[est_s:est_e+1].values
    x = mkt_ret.iloc[est_s:est_e+1].values
    valid = ~(np.isnan(y) | np.isnan(x))
    X = sm.add_constant(x[valid])
    ols = sm.OLS(y[valid], X).fit()
    a, b = ols.params

    y_ev = firm_ret.iloc[ev_s:ev_e+1].values
    x_ev = mkt_ret.iloc[ev_s:ev_e+1].values
    ar = y_ev - (a + b * x_ev)

    ar_results[tick] = {
        'name': row['name'], 'country': row['country'],
        'ar': ar, 'car': np.cumsum(ar),
        'car_total': np.nansum(ar),
    }

print(f"\nProcessed {len(ar_results)} firms across {len(unique_markets)} markets")
for tick, r in ar_results.items():
    print(f"  {tick:10s} ({r['country']:>15s}): CAR[-5,+5] = {r['car_total']*100:+.2f}%")

## 2. Intraday Event Studies

### 2.1 Motivation

Some events have a precise timestamp: FOMC announcements are released at 14:00 EST, earnings reports are filed with the SEC at known times, and court rulings have exact announcement moments. When the event time is known to the minute, using daily returns wastes information: the pre-announcement portion of the trading day is noise that dilutes the signal.

Intraday event studies use returns computed over intervals of minutes or hours around the announcement. The advantages are a sharper signal-to-noise ratio (the event window is shorter, so there is less noise) and the ability to study the dynamics of price adjustment at high frequency.

Barclay and Litzenberger (1988) pioneered the intraday event study in their analysis of new equity issue announcements. Bernanke and Kuttner (2005) used intraday data to measure the stock market's response to Federal Reserve policy surprises in narrow 30-minute windows around FOMC announcements.

### 2.2 Methodology

The intraday event study replaces the daily market model with an intraday benchmark. The simplest approach uses the market return over the same intraday interval as the benchmark:

$$
AR_{i,[t_a, t_b]} = R_{i,[t_a, t_b]} - \beta_i R_{m,[t_a, t_b]}
$$

where $R_{i,[t_a, t_b]}$ is the log return of firm $i$ from time $t_a$ to $t_b$, and $\beta_i$ is estimated from historical intraday data over corresponding intervals on non-event days.

In practice, many researchers use the market-adjusted return (setting $\beta_i = 1$) for intraday studies, because estimating intraday betas requires a large amount of high-frequency data and the reduction in variance from using the market model is modest over short intervals (the market return over 30 minutes is small relative to firm-specific returns).

The choice of event window is critical. A very narrow window (e.g., $\pm 5$ minutes) captures only the immediate reaction but may miss price discovery if the market takes time to process the information. A wider window (e.g., $\pm 30$ minutes or $\pm 60$ minutes) captures the full adjustment but introduces more noise. Bernanke and Kuttner (2005) used a 30-minute window (10 minutes before to 20 minutes after the FOMC announcement) and found that it captured most of the daily price movement.

### 2.3 Implementation

We demonstrate the intraday approach using simulated high-frequency data, because downloading live intraday data requires specialized (often paid) data feeds. The simulation generates 1-minute returns for a set of firms around a hypothetical FOMC announcement at 14:00 EST, with an injected abnormal return at the announcement time.

In [ ]:
# -- Simulated Intraday Event Study --
np.random.seed(2024)

# Simulate 1-minute returns for 390 minutes (full trading day, 9:30 to 16:00)
n_firms = 8
n_minutes = 390
announcement_minute = 270  # 14:00 EST = minute 270 (counting from 9:30)
inject_ar = 0.005  # 0.5% abnormal return at announcement

# Simulate: each firm has i.i.d. 1-min returns with sigma ~ 0.1% per minute
sigma_1min = 0.001
firm_returns_hf = np.random.normal(0, sigma_1min, (n_minutes, n_firms))
market_returns_hf = np.random.normal(0, sigma_1min * 0.7, n_minutes)

# Inject event: add abnormal return at announcement and next 5 minutes
for t in range(announcement_minute, min(announcement_minute + 5, n_minutes)):
    weight = 1.0 if t == announcement_minute else 0.3 * np.exp(-(t - announcement_minute) * 0.5)
    firm_returns_hf[t, :] += inject_ar * weight

minutes = np.arange(n_minutes)
times = pd.date_range('2024-01-31 09:30', periods=n_minutes, freq='min')

# Compute CARs in different windows around announcement
windows_intraday = [
    ('[-5min, +5min]', announcement_minute - 5, announcement_minute + 5),
    ('[-15min, +15min]', announcement_minute - 15, announcement_minute + 15),
    ('[-30min, +30min]', announcement_minute - 30, announcement_minute + 30),
    ('Full day', 0, n_minutes - 1),
]

print("Intraday Event Study: Simulated FOMC Announcement")
print("=" * 65)
for label, t_start, t_end in windows_intraday:
    t_s = max(0, t_start)
    t_e = min(n_minutes - 1, t_end)
    ar_window = firm_returns_hf[t_s:t_e+1, :] - market_returns_hf[t_s:t_e+1, np.newaxis]
    cars_hf = ar_window.sum(axis=0)
    mean_car = cars_hf.mean() * 100
    se_car = cars_hf.std(ddof=1) / np.sqrt(n_firms) * 100
    t_stat = mean_car / se_car
    print(f"  {label:20s}: CAAR = {mean_car:+.4f}%,  t = {t_stat:.3f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: Cumulative intraday returns
ax = axes[0]
for j in range(n_firms):
    cum_ret = np.cumsum(firm_returns_hf[:, j] - market_returns_hf) * 100
    ax.plot(minutes, cum_ret, alpha=0.4, linewidth=0.8)
avg_cum = np.mean([np.cumsum(firm_returns_hf[:, j] - market_returns_hf) for j in range(n_firms)], axis=0) * 100
ax.plot(minutes, avg_cum, 'k-', linewidth=2.5, label='Average')
ax.axvline(announcement_minute, color='red', linewidth=2, linestyle='--', alpha=0.7, label='Announcement (14:00)')
ax.set_xlabel('Minute of Trading Day')
ax.set_ylabel('Cumulative AR (%)')
ax.set_title('Panel A: Intraday Cumulative Abnormal Returns')
xtick_pos = [0, 60, 120, 180, 240, 270, 300, 360, 389]
xtick_labels = ['9:30', '10:30', '11:30', '12:30', '13:30', '14:00', '14:30', '15:30', '16:00']
ax.set_xticks(xtick_pos)
ax.set_xticklabels(xtick_labels, fontsize=8, rotation=30)
ax.legend(fontsize=9, frameon=True)
ax.grid(True, alpha=0.2)

# Panel B: Signal-to-noise by window width
ax = axes[1]
widths = np.arange(2, 120, 2)
snr = []
for w in widths:
    t_s = max(0, announcement_minute - w // 2)
    t_e = min(n_minutes - 1, announcement_minute + w // 2)
    ar_w = firm_returns_hf[t_s:t_e+1, :] - market_returns_hf[t_s:t_e+1, np.newaxis]
    cars_w = ar_w.sum(axis=0)
    t_val = cars_w.mean() / (cars_w.std(ddof=1) / np.sqrt(n_firms)) if cars_w.std() > 0 else 0
    snr.append(abs(t_val))
ax.plot(widths, snr, '-', color='#1f78b4', linewidth=2)
ax.axhline(1.96, color='red', linestyle='--', linewidth=1, alpha=0.5, label='t = 1.96')
ax.set_xlabel('Window Width (minutes)')
ax.set_ylabel('|t-statistic|')
ax.set_title('Panel B: Signal-to-Noise Ratio vs. Window Width')
ax.legend(fontsize=9, frameon=True)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 3. Confounding Events

### 3.1 The Problem

A confounding event is any event other than the focal event that occurs during the event window and affects the firm's stock price. If the event window is $[-1, +1]$, and the firm also announces a dividend change on day $+1$, the measured CAR reflects the combined impact of both events, and it is impossible to disentangle the two.

McWilliams and Siegel (1997) argued that confounding events are the most underappreciated threat to the validity of event studies. They reviewed a sample of published event studies and found that many failed to screen for confounding events, raising questions about the interpretation of their results.

### 3.2 Types of Confounding Events

Confounding events fall into three categories, each requiring a different treatment.

**Firm-specific confounders.** These include earnings announcements, dividend changes, management turnover, litigation developments, and other corporate news that coincide with the focal event. They can be identified by screening news databases (Factiva, LexisNexis) or SEC filings (8-K, 10-Q) during the event window. The standard approach is to exclude firms with confounders from the sample and re-estimate the CAAR. This reduces the sample size but eliminates the contamination.

**Market-wide confounders.** A macroeconomic announcement (employment report, Fed decision) that occurs during the event window affects all firms simultaneously. Unlike firm-specific confounders, market-wide confounders cannot be eliminated by excluding firms, because they affect the entire sample. The market model partially absorbs market-wide shocks (they are captured by $\beta_i R_{m,t}$), but the absorption is imperfect if the event firms have unusual factor exposures.

**Industry confounders.** An industry-level shock (regulatory ruling, commodity price movement) affects all firms in the same industry. If the sample is concentrated in one industry, the measured CAAR may reflect the industry shock rather than the focal event. Adding an industry return as a second factor in the normal return model mitigates this problem.

### 3.3 A Screening Protocol

We implement a systematic screening procedure. For each firm in the sample, we search for news items during the event window and flag potential confounders. We then re-estimate the CAAR with and without the flagged firms and report both estimates.

In [ ]:
# -- Confounding Event Analysis --
# Simulated example: screen for confounders in our Session 3-5 earnings sample

# Reuse the earnings sample
events_earnings = pd.DataFrame({
    'ticker': ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'JPM', 'JNJ', 'V', 'PG'],
    'event_date': pd.to_datetime([
        '2024-02-01', '2024-01-30', '2024-01-30', '2024-02-01',
        '2024-02-01', '2024-02-21', '2024-01-12', '2024-01-23',
        '2024-01-25', '2024-01-23'
    ]),
})

# Simulated confounder flags (in practice, these come from news screening)
confounders = {
    'AAPL': None,
    'MSFT': None,
    'GOOGL': 'Alphabet also announced $70B buyback on same day',
    'AMZN': None,
    'META': None,
    'NVDA': 'NVIDIA also announced new chip architecture same week',
    'JPM': None,
    'JNJ': 'J&J also announced talc litigation settlement during window',
    'V': None,
    'PG': None,
}

flagged = [t for t, c in confounders.items() if c is not None]
clean = [t for t, c in confounders.items() if c is None]

print("Confounding Event Screen")
print("=" * 70)
for tick, conf in confounders.items():
    status = f"FLAGGED: {conf}" if conf else "Clean"
    print(f"  {tick:6s}: {status}")

print(f"\nFlagged firms: {len(flagged)} ({', '.join(flagged)})")
print(f"Clean firms:   {len(clean)} ({', '.join(clean)})")

# Simulate impact on CAAR (using fake CARs for illustration)
np.random.seed(42)
cars_all = {t: np.random.normal(0.02, 0.04) for t in events_earnings['ticker']}
# Make confounded firms have larger CARs (they include the confounding effect)
for t in flagged:
    cars_all[t] += np.random.uniform(0.01, 0.03)

caar_full = np.mean(list(cars_all.values())) * 100
caar_clean = np.mean([cars_all[t] for t in clean]) * 100

print(f"\nImpact on CAAR [-1,+1]:")
print(f"  Full sample (N={len(cars_all)}): CAAR = {caar_full:+.2f}%")
print(f"  Clean sample (N={len(clean)}):  CAAR = {caar_clean:+.2f}%")
print(f"  Difference:                     {caar_full - caar_clean:+.2f}%")
print(f"\nReporting both estimates is standard practice. If the conclusions")
print(f"change qualitatively, the result is sensitive to confounders.")

## 4. The Conditional Event Study Framework

### 4.1 The Selection Problem

The standard event study treats the event as exogenous: it happens, and we measure the market's reaction. But many events are endogenous: the firm (or an agent) *chooses* whether and when to act. A firm decides whether to announce a dividend increase, whether to acquire a target, or whether to issue equity. If the market anticipates the decision, the announcement return reflects only the surprise component, not the full value impact.

Acharya (1988) and Prabhala (1997) formalized this problem. Consider a firm deciding whether to issue equity. Define:

$$
D_i = \begin{cases} 1 & \text{if firm } i \text{ issues equity} \\ 0 & \text{otherwise} \end{cases}
$$

The firm issues equity if the net benefit exceeds zero: $D_i = 1$ if $Z_i' \delta + u_i > 0$, where $Z_i$ is a vector of firm characteristics and $u_i$ is unobserved. The market observes $Z_i$ but not $u_i$. If the market uses $Z_i$ to forecast the probability of issuance, the announcement return reflects only the surprise:

$$
CAR_i = \gamma_0 + \gamma_1 \left(D_i - P(D_i = 1 \mid Z_i)\right) + \eta_i
$$

where $P(D_i = 1 \mid Z_i)$ is the market's prior probability. Firms that were expected to issue (high $P$) generate a small announcement return; firms that were not expected to issue (low $P$) generate a large return.

### 4.2 The Two-Stage Estimation

The conditional event study is estimated in two stages:

**Stage 1: Selection equation.** Estimate a probit (or logit) model of the event decision on a sample that includes both event firms and non-event firms (the "full choice set"):

$$
P(D_i = 1 \mid Z_i) = \Phi(Z_i' \hat{\delta})
$$

where $\Phi$ is the standard normal CDF. This stage produces the predicted probability $\hat{p}_i = \Phi(Z_i' \hat{\delta})$ for each firm.

**Stage 2: Outcome equation.** For the event firms only, regress the CAR on a function of $\hat{p}_i$. The simplest specification is:

$$
CAR_i = \gamma_0 + \gamma_1 \hat{p}_i + \eta_i
$$

A negative $\gamma_1$ indicates that firms with higher prior probability of the event (i.e., less surprising announcements) have lower CARs, which is consistent with the market partially anticipating the event.

Prabhala (1997) showed that the standard (unconditional) event study is valid under certain conditions (specifically, when the selection equation is correctly specified and the error terms $u_i$ and $\eta_i$ are independent). When these conditions fail, the conditional approach produces more efficient estimates.

### 4.3 Implementation

We illustrate the conditional framework with a dividend increase sample. In the first stage, we model the probability of a dividend increase using firm characteristics (profitability, payout ratio, prior dividend growth). In the second stage, we test whether the announcement return is larger for firms with a lower predicted probability of the increase (i.e., more surprising announcements).

In [ ]:
# -- Conditional Event Study: Simulated Dividend Increase --
np.random.seed(123)

N_full = 200  # full choice set: 200 firms
N_event = 60  # 60 firms increase dividends

# Stage 1 data: firm characteristics
Z = pd.DataFrame({
    'profitability': np.random.normal(0.10, 0.04, N_full),     # ROE
    'payout_ratio': np.random.uniform(0.2, 0.8, N_full),       # dividend/earnings
    'prior_div_growth': np.random.normal(0.03, 0.05, N_full),  # last year's div growth
})

# Latent variable: firms with high profitability and low payout are more likely to increase
latent = 0.5 + 3 * Z['profitability'] - 1.5 * Z['payout_ratio'] + 2 * Z['prior_div_growth']
latent += np.random.normal(0, 1, N_full)
Z['increase'] = (latent > np.percentile(latent, 100 * (1 - N_event / N_full))).astype(int)

# Stage 1: Probit
from statsmodels.discrete.discrete_model import Probit
X_probit = sm.add_constant(Z[['profitability', 'payout_ratio', 'prior_div_growth']])
probit_model = Probit(Z['increase'], X_probit).fit(disp=0)
Z['p_hat'] = probit_model.predict(X_probit)

print("Stage 1: Probit Model of Dividend Increase Decision")
print("=" * 60)
print(probit_model.summary2().tables[1].to_string())

# Generate CARs for event firms: CAR depends on surprise (1 - p_hat)
event_firms = Z[Z['increase'] == 1].copy()
event_firms['CAR'] = 0.005 + 0.03 * (1 - event_firms['p_hat']) + np.random.normal(0, 0.02, len(event_firms))
event_firms['CAR_pct'] = event_firms['CAR'] * 100

# Stage 2: Outcome regression
X_stage2 = sm.add_constant(event_firms[['p_hat']])
stage2 = sm.OLS(event_firms['CAR_pct'], X_stage2).fit(cov_type='HC1')

print(f"\nStage 2: CAR on Predicted Probability (N = {len(event_firms)})")
print("=" * 60)
print(stage2.summary2().tables[1].to_string())
print(f"\nInterpretation: a coefficient of {stage2.params['p_hat']:.2f} on p_hat means that")
print(f"  firms with a 10pp higher prior probability of increasing dividends")
print(f"  have {stage2.params['p_hat'] * 0.10:.2f}pp lower CARs (less surprise).")

In [ ]:
# -- Visualization of the conditional relationship --
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Panel A: Selection model - predicted probabilities
ax = axes[0]
ax.hist(Z.loc[Z['increase'] == 0, 'p_hat'], bins=25, alpha=0.6, color='#1f78b4',
        density=True, label='Non-event firms', edgecolor='white')
ax.hist(Z.loc[Z['increase'] == 1, 'p_hat'], bins=25, alpha=0.6, color='#d95f02',
        density=True, label='Event firms (div increase)', edgecolor='white')
ax.set_xlabel('Predicted Probability of Dividend Increase ($\hat{p}_i$)')
ax.set_ylabel('Density')
ax.set_title('Panel A: Selection Model Fit')
ax.legend(fontsize=9, frameon=True)
ax.grid(True, alpha=0.2)

# Panel B: CAR vs predicted probability
ax = axes[1]
ax.scatter(event_firms['p_hat'], event_firms['CAR_pct'], s=40, color='#1f78b4', alpha=0.6, zorder=3)
x_line = np.linspace(event_firms['p_hat'].min(), event_firms['p_hat'].max(), 100)
y_line = stage2.params['const'] + stage2.params['p_hat'] * x_line
ax.plot(x_line, y_line, 'r-', linewidth=2,
        label=f'$\\gamma_1$ = {stage2.params["p_hat"]:.2f} (t = {stage2.tvalues["p_hat"]:.2f})')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Predicted Probability ($\hat{p}_i$)')
ax.set_ylabel('CAR (%)')
ax.set_title('Panel B: Announcement Return vs. Prior Probability')
ax.legend(fontsize=9, frameon=True)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 5. Choosing the Right Event Study Design: A Decision Tree

The extensions discussed in this session, together with the standard methods of Sessions 1 through 7, form a toolkit that the researcher must adapt to the specific research question. The following decision tree summarizes the key design choices.

**What is the event horizon?** If the event window is a few days, use the standard short-horizon methodology (Sessions 1-5). If the window is months or years, use the long-horizon methods of Session 7 (BHAR, calendar-time portfolio).

**Is the event time known to the minute?** If yes, and if high-frequency data is available, consider an intraday event study (Section 2). This is particularly valuable for monetary policy announcements, earnings releases with precise timestamps, and court rulings.

**Are the firms in multiple countries?** If yes, adjust event dates for time zones, use local market indices as benchmarks, and compute abnormal returns in local currency (Section 1).

**Are there potential confounding events?** Screen for confounders using news databases. Report results with and without flagged firms (Section 3).

**Is the event endogenous?** If the event is a corporate decision (issuance, acquisition, dividend change), consider the conditional event study framework (Section 4) to account for the market's partial anticipation of the decision.

**What explains the cross-sectional variation in CARs?** Use the cross-sectional regression of Session 6 to relate firm-level CARs to observable characteristics.

**Which test statistics to report?** At minimum, report the BMP test (Session 4) and the Corrado rank test (Session 5) for short-horizon studies. For long-horizon studies, report both event-time BHARs and calendar-time portfolio alphas (Session 7).

## 6. Summary and Preview of Session 9

This session extended the event study methodology in four directions: multi-country studies, intraday analysis, confounding event treatment, and the conditional event study framework.

The multi-country extension requires careful handling of time zones, local benchmarks, and currency effects, but the cross-sectional aggregation and testing proceed as in the single-country case. The intraday extension exploits precise event timing to increase the signal-to-noise ratio, with the optimal window width trading off signal capture against noise accumulation. Confounding events must be screened systematically and their impact assessed by comparing results with and without flagged observations. The conditional framework addresses the endogeneity of the event decision by modelling the market's prior probability and testing whether the announcement return depends on the degree of surprise.

Session 9 brings together the full toolkit in a practical guide to designing, implementing, and reporting an event study from start to finish. It covers research question formulation, sample construction, implementation choices, robustness protocols, and publication standards. The session is organized as a checklist that the researcher can follow step by step.

**Additional references:**

- Eckbo, B.E., Masulis, R.W. and Norli, O. (2007). Security Offerings. In B.E. Eckbo (ed.), *Handbook of Empirical Corporate Finance*, Volume 1. Elsevier, Chapter 6.
- Khotari, S.P. and Warner, J.B. (2007). Econometrics of Event Studies. In B.E. Eckbo (ed.), *Handbook of Empirical Corporate Finance*, Volume 1. Elsevier, Chapter 1.
- Li, K. and Prabhala, N.R. (2007). Self-Selection Models in Corporate Finance. In B.E. Eckbo (ed.), *Handbook of Empirical Corporate Finance*, Volume 1. Elsevier, Chapter 2.